# 🎈✨🎉🎂 <span style="color: white; background-color: steelblue;"><b> Envio de E-mails Feliz Aniversário </b></span></p>

🧩 <span style="color: mediumseagreen;"><b> 1- Coleta de Dados e Configuração Inicial </b></span></p>
- A automação utiliza três fontes principais:
    - Controle_HC e Atestados.xlsx → dados completos dos colaboradores  
    - EMAIL.xlsx → base de e‑mails corporativos  
    - Imagem de aniversário → arquivo que será embutido no e‑mail
- E também configura:
    - Conta de envio: gestaodepessoas@afpesp.org.br  
    - Pasta de destino dos relatórios: 10.10 – Aniversários  
    - Lista de colunas finais

📅 <span style="color: mediumseagreen;"><b> 2- Cálculo e Tratamento das Datas de Aniversário </b></span></p>
- O processo executa conversão da data de nascimento
- Ajuste para formatos heterogêneos (string, texto, Excel serial)
- Correção de anos bissextos:
    - Se houver 29/02 em um ano não bissexto → corrige para 28/02
- Cálculo da próxima data de aniversário:
    - Baseado na data atual e no ano corrente
- O usuário escolhe a data de início, e o sistema calcula:
    - Quem faz aniversário entre essa data e +7 dias? (ou outro intervalo conforme configurado)

🔍 <span style="color: mediumseagreen;"><b> 3- Filtragem e Seleção dos Colaboradores </b></span></p>
- A automação:
    - Mantém apenas colaboradores ativos  
    - Seleciona apenas as colunas essenciais  
    - Identifica aniversariantes na janela de tempo  
    - Calcula idade automaticamente  
    - Ordena pela data de aniversário

📤 <span style="color: mediumseagreen;"><b> 4- Geração do Relatório de Aniversariantes </b></span></p>
- O sistema gera Aniversariantes_YYYYMMDD.xlsx
- Com colunas:
    - Matrícula  
    - Nome  
    - Nascimento  
    - Idade  
    - Sexo  
    - Cargo  
    - Centro de custo  
    - Situação  
    - Unidade / Empresa  
    - Data de admissão
- O relatório é salvo diretamente na pasta destino

🔗 <span style="color: mediumseagreen;"><b> 5- Enriquecimento com E‑mails </b></span></p>
- Para garantir que todos os aniversariantes têm contato válido:
    - Carrega EMAIL.xlsx
    - Valida colunas necessárias
    - Realiza merge por nome
    - Preenche a coluna “E-mail”
- Após essa etapa, o script exibe um modal de confirmação:
    - “Todos os aniversariantes possuem e-mail válido?”
    - O processo só continua após confirmação

✉️ <span style="color: mediumseagreen;"><b> 6- Envio Automático dos E‑mails Personalizados </b></span></p>
- Cada aniversariante recebe um e‑mail individual contendo:
    - Saudação personalizada (ex.: “Olá, João!”)  
    - Mensagem comemorativa em HTML  
    - Imagem embutida via Content-ID (CID)  
    - Envio pela caixa compartilhada “gestaodepessoas@afpesp.org.br”
- O corpo do e‑mail é montado via HTML + Outlook COM API
- O envio utiliza:
    - win32com.client (Outlook automation)  
    - PyAutoGUI para pop-ups  
    - Regex para validar e‑mails  
    - Registro de sucesso, falha e e‑mails inválidos

📊 <span style="color: mediumseagreen;"><b> 7- Relatório Final da Execução </b></span></p>
- E‑mails enviados com sucesso  
- Falhas de envio  
- E‑mails inválidos  
- Total processado  
- Tempo total de execução
- Isso possibilita uma visão clara de performance e traz segurança operacional

# Criando o Relatório de Aniversariantes

In [1]:
import logging
import os
import pandas as pd
import pyautogui
import re
import shutil
import sys
import win32com.client as win32
from asyncio.log import logger
from datetime import datetime, date, timedelta
from openpyxl import load_workbook, Workbook
from pathlib import Path

tempo_0 = [id, datetime.today(), 0]

# === CONFIG ===
CAMINHO_ARQUIVO = r'X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx'
CAMINHO_EMAIL = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\EMAIL.xlsx'
CONTA_ENVIO = 'gestaodepessoas@afpesp.org.br'
IMAGE_PATH = r'X:\Gestão de Pessoas\Analytics\01 - Templates\01.0 - Imagens\art_aniversario.jpg'
PASTA_DESTINO = r'X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.10 - Aniversários'
PROCESSOS_PATH = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'
id = 30

COLUNAS_FINAIS = [
    'registro', 'nome', 'nascimento', 'sexo', 'idade',
    'data_admissao', 'situacao', 'cargo_completo', 'centro_custo', 'empresa_resumo'
]

# === FUNÇÕES AUXILIARES ===
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

def is_leap_year(year):
    """Retorna True se o ano for bissexto."""
    return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

def converter_data_nascimento(valor):
    """Converte valor para datetime, aceita serial Excel, datetime ou string.
    Retorna None se inválido ou fora do intervalo 1950-hoje."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    # Serial Excel
    if isinstance(valor, (int, float)) and valor > 1:
        try:
            dt = pd.to_datetime(valor, unit='D', origin='1899-12-30')
        except:
            return None
    elif isinstance(valor, datetime):
        dt = valor
    else:
        # Tenta converter string
        try:
            dt = pd.to_datetime(valor, dayfirst=True, errors='coerce')
        except:
            return None
    if pd.isna(dt):
        return None
    # Verifica intervalo válido
    if dt.year < 1950 or dt.date() > date.today():
        return None
    return dt

def obter_data_inicio():
    """Solicita data no formato DD/MM/AAAA e retorna objeto date."""
    while True:
        entrada = input("Digite a data de início (DD/MM/AAAA): ").strip()
        try:
            dia, mes, ano = map(int, entrada.split('/'))
            data = date(ano, mes, dia)
            return data
        except:
            print("Formato inválido. Use DD/MM/AAAA.")

def carregar_dados():
    """Carrega a planilha"""
    try:
        df = pd.read_excel(CAMINHO_ARQUIVO, sheet_name='HC', engine='openpyxl')
        return df
    except Exception as e:
        print(f"Erro ao carregar arquivo: {e}")
        sys.exit(1)

def filtrar_ativos(df):
    """Filtra apenas registros com situacao == 'ATIVO'."""
    return df[df['status'] == 'ATIVO'].copy()

def selecionar_colunas(df):
    """Seleciona apenas as colunas finais do relatório."""
    return df[COLUNAS_FINAIS].copy()

def converter_datas(df):
    """Converte colunas de data para formato correto.
    Cria data_nascimento_temp (datetime, mantém para cálculos).
    Formata nascimento e data_admissao como DD/MM/AAAA (string)."""
    
    # Converter nascimento
    if 'nascimento' in df.columns:
        df['data_nascimento_temp'] = df['nascimento'].apply(converter_data_nascimento)
        df['nascimento'] = df['data_nascimento_temp'].dt.strftime('%d/%m/%Y')
    
    # Converter data_admissao com a mesma função robusta
    if 'data_admissao' in df.columns:
        df['data_admissao_temp'] = df['data_admissao'].apply(converter_data_nascimento)
        df['data_admissao'] = df['data_admissao_temp'].dt.strftime('%d/%m/%Y')
        df.drop(columns=['data_admissao_temp'], inplace=True)
    
    return df

def ajustar_fevereiro_29(df):
    """Ajusta dia 29/02 para 28/02 em anos não bissextos.
    Deve ser chamado antes de qualquer replace() de ano."""
    mask = (df['data_nascimento_temp'].dt.month == 2) & (df['data_nascimento_temp'].dt.day == 29)
    anos = df.loc[mask, 'data_nascimento_temp'].dt.year
    for idx in df[mask].index:
        ano = df.at[idx, 'data_nascimento_temp'].year
        if not is_leap_year(ano):
            df.at[idx, 'data_nascimento_temp'] = df.at[idx, 'data_nascimento_temp'].replace(day=28)
    return df

def prox_aniv_calc(row):
    """Calcula o próximo aniversário a partir da data_nascimento_temp."""
    hoje = date.today()
    nasc = row['data_nascimento_temp']
    if nasc is None or pd.isna(nasc):
        return None
    try:
        prox = nasc.replace(year=hoje.year)
        if prox < hoje:
            prox = prox.replace(year=hoje.year + 1)
        return prox
    except:
        return None

def filtrar_aniversarios(df, data_inicio):
    """Filtra aniversariantes cujo aniversário ocorre entre data_inicio e hoje."""
    hoje = date.today()
    
    def aniversario_no_periodo(data_nasc):
        """Verifica se o aniversário (mes-dia) cai entre data_inicio e hoje."""
        if pd.isna(data_nasc):
            return False
        
        # Extrai mes-dia como tupla (mes, dia)
        nasc_mes_dia = (data_nasc.month, data_nasc.day)
        inicio_mes_dia = (data_inicio.month, data_inicio.day)
        fim_mes_dia = (hoje.month, hoje.day)
        
        # Se intervalo NÃO cruza o ano (ex: maio para maio)
        if inicio_mes_dia <= fim_mes_dia:
            return inicio_mes_dia <= nasc_mes_dia <= fim_mes_dia
        else:
            # Se intervalo CRUZA o ano (ex: dezembro para janeiro)
            return nasc_mes_dia >= inicio_mes_dia or nasc_mes_dia <= fim_mes_dia
    
    # Filtra linhas onde aniversário está no período
    mask = df['data_nascimento_temp'].apply(aniversario_no_periodo)
    df_filtrado = df[mask].copy()
    
    # Criar coluna auxiliar para ordenação
    df_filtrado['aniversario_ref'] = df_filtrado['data_nascimento_temp'].dt.strftime('%m-%d')
    df_filtrado['prox_aniv'] = df_filtrado['data_nascimento_temp'].dt.strftime('%m-%d')
    
    return df_filtrado

def ordenar_por_aniversario(df):
    """Ordena o DataFrame pela data do próximo aniversário."""
    return df.sort_values('prox_aniv')

def exportar_resultado(df):
    """Remove colunas temporárias, reordena colunas finais e salva .xlsx."""
    df = df.drop(columns=['data_nascimento_temp', 'aniversario_ref', 'prox_aniv'], errors='ignore')
    df = df[COLUNAS_FINAIS]
    hoje_str = date.today().strftime('%Y%m%d')
    nome_arquivo = f'Aniversariantes_{hoje_str}.xlsx'
    caminho_completo = os.path.join(PASTA_DESTINO, nome_arquivo)
    try:
        df.to_excel(caminho_completo, index=False)
        print(f"Arquivo salvo em: {caminho_completo}")
    except Exception as e:
        print(f"Erro ao salvar arquivo: {e}")

def main():
    print("=== GERADOR DE LISTA DE ANIVERSARIANTES ===")
    data_inicio = obter_data_inicio()
    df = carregar_dados()
    df = filtrar_ativos(df)
    df = selecionar_colunas(df)
    df = converter_datas(df)
    df = filtrar_aniversarios(df, data_inicio)
    if df.empty:
        print("Nenhum aniversariante encontrado no período.")
        return
    df = ordenar_por_aniversario(df)
    exportar_resultado(df)
    print("Processo concluído com sucesso.")

if __name__ == '__main__':
    main()

=== GERADOR DE LISTA DE ANIVERSARIANTES ===
Arquivo salvo em: X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.10 - Aniversários\Aniversariantes_20260625.xlsx
Processo concluído com sucesso.


# Localizando o E-mail do Aniversariante

In [2]:
def carregar_aniversariantes(caminho):
    try:
        hoje = date.today().strftime('%Y%m%d')
        nome_arquivo = f'Aniversariantes_{hoje}.xlsx'
        caminho_completo = os.path.join(caminho, nome_arquivo)
        df = pd.read_excel(caminho_completo)
        return df, caminho_completo
    except FileNotFoundError as e:
        print(f'Erro: Arquivo de aniversariantes não encontrado em {caminho_completo}. Detalhe: {e}')
        sys.exit(1)
    except Exception as e:
        print(f'Erro ao carregar aniversariantes: {e}')
        sys.exit(1)

def carregar_email(caminho):
    try:
        df = pd.read_excel(caminho)
        return df
    except FileNotFoundError as e:
        print(f'Erro: Arquivo EMAIL.xlsx não encontrado em {caminho}. Detalhe: {e}')
        sys.exit(1)
    except Exception as e:
        print(f'Erro ao carregar email: {e}')
        sys.exit(1)

def validar_colunas(df_aniversariantes, df_email):
    if 'nome' not in df_aniversariantes.columns:
        print('Erro: Coluna "nome" não encontrada na planilha de aniversariantes.')
        sys.exit(1)
    if 'nome' not in df_email.columns or 'email' not in df_email.columns:
        print('Erro: Colunas "nome" e "email" devem existir na planilha EMAIL.')
        sys.exit(1)

def merge_dados(df_aniversariantes, df_email):
    try:
        df_email_subset = df_email[['nome', 'email']]
        df_merged = pd.merge(df_aniversariantes, df_email_subset, on='nome', how='left')
        return df_merged
    except Exception as e:
        print(f'Erro ao realizar merge: {e}')
        sys.exit(1)

def salvar_arquivo(df, caminho):
    try:
        df.to_excel(caminho, index=False)
        print(f'Arquivo salvo com sucesso em {caminho}')
    except Exception as e:
        print(f'Erro ao salvar arquivo: {e}')
        sys.exit(1)

def main():
    # Carregar dados
    df_aniversariantes, caminho_aniversario = carregar_aniversariantes(PASTA_DESTINO)
    df_email = carregar_email(CAMINHO_EMAIL)

    # Validar colunas
    validar_colunas(df_aniversariantes, df_email)

    # Realizar merge
    df_resultado = merge_dados(df_aniversariantes, df_email)

    # Salvar (sobrescrever)
    salvar_arquivo(df_resultado, caminho_aniversario)

if __name__ == '__main__':
    main()

Arquivo salvo com sucesso em X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.10 - Aniversários\Aniversariantes_20260625.xlsx


# Conferência se o Arquivo possui todos os E-mails dos Aniversariantes

In [3]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o arquivo estão com e-mails dos colaboradores cadastrados.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Todos os colaboradores estão com o e-mail cadastrado?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-25 09:09:17,860 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-25 09:09:18,187 - INFO - Iniciando a leitura e processamento dos dados...


# Enviando E-mail

In [4]:
# Verifica se os arquivos necessários existem
def verificar_arquivos(EXCEL_PATH, IMAGE_PATH):
    if not os.path.exists(EXCEL_PATH):
        raise FileNotFoundError(f"Arquivo Excel não encontrado: {EXCEL_PATH}")
    if not os.path.exists(IMAGE_PATH):
        raise FileNotFoundError(f"Imagem não encontrada: {IMAGE_PATH}")

# Valida formato básico de e-mail
def email_valido(email):
    return bool(re.match(r'^[^@\s]+@[^@\s]+\.[^@\s]+$', str(email).strip()))

# Lê o arquivo Excel e retorna o DataFrame
def ler_excel(pasta_destino):
    """Localiza e lê o arquivo Aniversariantes_YYYYMMDD.xlsx da pasta."""
    try:
        # Listar arquivos na pasta
        arquivos = os.listdir(pasta_destino)
        
        # Procurar por arquivo que começa com 'Aniversariantes_'
        arquivo_encontrado = None
        for arquivo in arquivos:
            if arquivo.startswith('Aniversariantes_') and arquivo.endswith('.xlsx'):
                arquivo_encontrado = arquivo
                break
        
        if arquivo_encontrado is None:
            print(f"Erro: Nenhum arquivo 'Aniversariantes_*.xlsx' encontrado em {pasta_destino}")
            return None
        
        # Construir caminho completo
        caminho_completo = os.path.join(pasta_destino, arquivo_encontrado)
        
        # Ler o arquivo
        df = pd.read_excel(caminho_completo)
        print(f"Arquivo lido: {arquivo_encontrado}")
        return df
        
    except Exception as e:
        print(f"Erro ao ler Excel: {e}")
        return None

# Cria instância do Outlook
def criar_outlook():
    try:
        outlook = win32.Dispatch("Outlook.Application")
        return outlook
    except Exception as e:
        print(f"Erro ao conectar com Outlook: {str(e)}. Verifique se o Outlook está instalado.")
        return None

# Monta o corpo HTML do e-mail
def montar_html(nome):
    return f"""
    <html>
    <body style="font-family: Arial, sans-serif;">
        <h2 style="color: #0070C0;">Olá, {nome}!</h2>
        <img src="cid:image_cid" alt="Arte de Aniversário" style="max-width: 100%; height: auto;">
        <p>Parabéns pelo seu dia!<br> É um privilégio contar com sua parceria e dedicação diária.<br> Que este novo ciclo seja repleto de realizações, saúde e sucesso.<br>
        Feliz aniversário!</p>
        <br>
        <p>Atenciosamente,<br>Equipe Gestão de Pessoas</p>
    </body>
    </html>
    """

# Envia e-mail com imagem embutida
def enviar_email(outlook, destinatario, nome, assunto, image_path):
    try:
        mail = outlook.CreateItem(0)  # 0 = olMailItem
        mail.To = destinatario
        mail.Subject = assunto

        # Configura remetente como caixa compartilhada (Shared Mailbox)
        # Shared Mailboxes não aparecem em Session.Accounts — usar CreateRecipient
        remetente = outlook.Session.CreateRecipient(CONTA_ENVIO)
        remetente.Resolve()
        if remetente.Resolved:
            mail.Sender = remetente.AddressEntry
            mail.SentOnBehalfOfName = CONTA_ENVIO
        else:
            print(f"ERRO: Não foi possível resolver '{CONTA_ENVIO}'. Verifique se você tem permissão de 'Enviar como' nessa caixa compartilhada.")
            return False

        # Anexa a imagem como inline (CID)
        attachment = mail.Attachments.Add(os.path.abspath(image_path))
        attachment.PropertyAccessor.SetProperty(
            "http://schemas.microsoft.com/mapi/proptag/0x3712001E",
            "image_cid"
        )

        mail.HTMLBody = montar_html(nome)
        mail.Save()   # Salva rascunho antes de enviar (evita perda em falhas)
        mail.Send()
        print(f"E-mail enviado com sucesso para {destinatario} ({nome})")
        return True

    except Exception as e:
        print(f"Erro ao enviar e-mail para {destinatario}: {str(e)}")
        return False

def main():
    try:
        verificar_arquivos(PASTA_DESTINO, IMAGE_PATH)
        df = ler_excel(PASTA_DESTINO)
        if df is None or df.empty:
            print("Nenhum dado válido encontrado no Excel.")
            return
            
        print(f"Lidos {len(df)} registros do Excel.")
        outlook = criar_outlook()
        if outlook is None:
            return
            
        assunto = "Feliz Aniversário!"
        sucessos = 0
        falhas = 0 
        invalidos = 0
        
        for _, row in df.iterrows():
            email = row['email'] 
            nome = row['nome']
            if not email_valido(email):
                print(f"E-mail inválido pulado: {email}")
                invalidos += 1
                continue 
            if enviar_email(outlook, email, nome, assunto, IMAGE_PATH):
                sucessos += 1
            else: 
                falhas += 1
                
        print("\n" + "=" * 50)
        print("RELATÓRIO DE EXECUÇÃO:") 
        print(f"   Enviados com sucesso : {sucessos}")
        print(f"   Falhas de envio      : {falhas}") 
        print(f"   E-mails inválidos    : {invalidos}")
        print(f"   Total processados    : {sucessos + falhas + invalidos}")
        print("=" * 50) 

        # === MOVIMENTAÇÃO DO ARQUIVO ===
        print("\nMovendo o arquivo processado...")
        
        # Define o caminho da nova pasta
        pasta_movidos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'
        
        # Garante que a pasta de destino existe, se não existirá, ela é criada
        if not os.path.exists(pasta_movidos):
            os.makedirs(pasta_movidos)
            
        # Como o nome do arquivo muda conforme a data, listamos a pasta para achar o arquivo correto igual você fez no 'ler_excel'
        arquivos = os.listdir(PASTA_DESTINO)
        arquivo_para_mover = None
        
        for arquivo in arquivos:
            if arquivo.startswith('Aniversariantes_') and arquivo.endswith('.xlsx'):
                arquivo_para_mover = arquivo 
                break
                
        if arquivo_para_mover:
            origem = os.path.join(PASTA_DESTINO, arquivo_para_mover)
            destino = os.path.join(pasta_movidos, arquivo_para_mover)
            
            # Executa a movimentação (recorta da origem e cola no destino)
            shutil.move(origem, destino)
            print(f" ✅  Arquivo '{arquivo_para_mover}' movido com sucesso para: {pasta_movidos}")
        else:
            print(" ⚠️  Aviso: Nenhum arquivo 'Aniversariantes_*.xlsx' foi encontrado para ser movido.")
        # ===========================================

    except Exception as e:
        print(f"Erro geral no script: {str(e)}")

def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operações seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)
        
def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    with gerenciar_workbook(caminho, 'REGISTROS') as ws:
        ws.append([id_proc, datetime.today(), etapa])
    logger.debug(f"✅ Etapa {etapa} registrada")

if __name__ == "__main__":
    main()

tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

Arquivo lido: Aniversariantes_20260625.xlsx
Lidos 5 registros do Excel.
E-mail enviado com sucesso para creomas@yahoo.com.br (CREOMAR SANTOS VILAS BOAS)
E-mail enviado com sucesso para leandroluizrocha37@gmail.com (LEANDRO LUIZ ROCHA DE MELLO)
E-mail enviado com sucesso para marcakalz01@gmail.com (LUCAS DOS SANTOS)
E-mail enviado com sucesso para fabiano.silva@afpesp.org.br (FABIANO LEITE DA SILVA)
E-mail enviado com sucesso para vihh.25cabral@gmail.com (VITORIA CABRAL GASTAO)

RELATÓRIO DE EXECUÇÃO:
   Enviados com sucesso : 5
   Falhas de envio      : 0
   E-mails inválidos    : 0
   Total processados    : 5

Movendo o arquivo processado...
 ✅  Arquivo 'Aniversariantes_20260625.xlsx' movido com sucesso para: X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS
----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:01:35.467139

--------------------------------------